
# 09 — Lifting the univariate constraint

**Why did the sequence models only ever see lags?**

That question has a specific answer, and it is not a modelling one. The original study
built its sequence models with `input_shape=(lookback, 1)` — a bare window of past
demand, nothing else. `bwalloc.sequence` reproduces that exactly, and notebook 08's
comparison depends on it: if the trees had been handed Fourier terms and context flags
while the sequence models got a bare window, the contrast would have measured the
feature set rather than the architecture.

So the univariate window is a **reproduction constraint**. It was never a recommendation,
and it has never been lifted — until this notebook.

Two facts make lifting it worth doing:

1. **`is_rain` is the one feature the audit found a real effect for** — residual variance
   ratio 1.330, Levene p < 0.001 on GP — and no sequence model in this project has ever
   been shown it.
2. **The tree models have the opposite problem.** Notebook 01 gives them Fourier terms
   and context flags but only four lags, and notebook 08 showed that costs 18% RMSE on
   GP. So **no model in this project has yet seen the full feature set at full lag
   depth.**

## Covariates are time-varying, and that changes how they enter

There is a value of `is_rain` at *each* of the 24 past timestamps, not one value for the
window. Flattening them to a single number throws most of that away. Following the
covariate taxonomy that DeepAR and the Temporal Fusion Transformer use, the design
matrix is read as three blocks:

| block | what it holds |
|---|---|
| **past covariates** | demand lags 1–24, each context flag at lags 1–24, daily Fourier at lags 1–24 — `(rows, 24, channels)` |
| **known at origin** | Fourier terms of the *target* timestamp, legitimately available because the clock is not something we forecast |
| **static** | rolling mean and standard deviation |

Channel 0 is always demand, so a model that ignores every other channel degenerates
exactly to notebook 08. That makes *"do the covariates earn anything?"* a measurable
quantity rather than an assumption — and `tests/test_bwalloc.py` fails any architecture
whose predictions do not move when a covariate channel does.

In [ ]:
# --- Bootstrap: works locally, on Colab and on Kaggle ----------------------
# RUN THIS CELL FIRST, and re-run it after any kernel restart. Every later cell
# depends on it. If it fails, the next cell fails with "No module named bwalloc",
# which looks like a different problem but is not.
#
# On a hosted runtime it clones the repo (or pulls, on a re-run) and installs the
# package into the session, so `import bwalloc` keeps working even from a cell you
# run on its own after restarting. The repository is public; no token is needed.
import os, subprocess, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
REPO = "https://github.com/sad-code-at/bwalloc.git"

def _find_root(start: Path):
    node = start
    while not (node / "src" / "bwalloc").exists() and node != node.parent:
        node = node.parent
    return node if (node / "src" / "bwalloc").exists() else None

ROOT = _find_root(Path.cwd())

if ROOT is None:
    # A repo uploaded as a Kaggle Dataset mounts read-only here; prefer it if present.
    for candidate in Path("/kaggle/input").glob("*/src/bwalloc"):
        ROOT = candidate.parent.parent
        break

if ROOT is None:
    on_kaggle = Path("/kaggle/working").exists()
    target = Path("/kaggle/working/bwalloc") if on_kaggle else Path("/content/bwalloc")
    if _find_root(target) is None:
        if os.system("git clone -q " + REPO + " " + str(target)) != 0 or _find_root(target) is None:
            raise RuntimeError(
                "Clone failed. On Kaggle, switch Internet ON in the right sidebar "
                "(Settings > Internet), then re-run this cell. See docs/KAGGLE.md."
            )
        print("cloned", REPO)
    else:
        os.system("git -C " + str(target) + " pull -q --ff-only")
        print("pulled latest into", target)
    os.chdir(target)
    ROOT = target
    # Install into the session so `import bwalloc` survives a kernel restart and
    # does not depend on this cell having set sys.path. --no-deps deliberately:
    # Kaggle and Colab curate their own numpy/pandas and we must not disturb them.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
                   cwd=str(ROOT), capture_output=True)

sys.path.insert(0, str(ROOT / "src"))

try:
    import xgboost  # noqa: F401
except ImportError:
    !pip install -q xgboost

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bwalloc as bw
from bwalloc.plots import use_paper_style

bw.set_seed()
use_paper_style()
pd.set_option("display.width", 200)
RESULTS = ROOT / "experiments" / "results"
# Scratch output for the exploratory notebooks. Only 07_paper_figures writes into
# paper/figures -- otherwise running notebook 00 or 05 silently overwrites a figure
# the paper cites, which is exactly the kind of drift this project exists to remove.
FIGURES = ROOT / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
_data = sorted((ROOT / "data").glob("*.csv"))
_res = sorted(RESULTS.glob("*.csv"))
print("bwalloc", bw.__version__, "at", ROOT)
print(f"  data/    {len(_data)} csv  ({', '.join(f.name for f in _data) or 'MISSING'})")
print(f"  results/ {len(_res)} csv")
if not _data:
    raise RuntimeError(
        "The trace CSVs are missing, so nothing will run. Re-run this cell to "
        "re-clone, or check that the repository was fetched completely."
    )


## The five arms

Every arm is scored on **one common row index**, so the fold schedule cannot differ
between them. Arms drop different numbers of warm-up rows — four wall-clock lags reach
back 17 samples on GP, a 24-lag window reaches back 24 — and comparing them on different
folds would confound the feature set with the split, which is the exact mistake the
original study made.

In [ ]:

from bwalloc.baselines import standard_baselines
from bwalloc.data import load, sampling_profile
from bwalloc.evaluate import beats_baseline, dm_matrix, run_backtest, summarise
from bwalloc.features import FeatureConfig, assert_no_leakage, build_features
from bwalloc.models import default_point_models
from bwalloc.sequence import (channel_window, covariate_sequence_models,
                              full_feature_config, sequence_feature_config,
                              sequence_models)
from bwalloc.splits import rolling_origin

OPERATOR = "gp"          # switch to "robi" and re-run
N_FOLDS, LOOKBACK, EPOCHS = 8, 24, 30

ARMS = {
    # name:                (config, has a dense demand window, has covariate channels)
    "lags_only":            (sequence_feature_config(LOOKBACK), True, False),
    "full_sparse":          (FeatureConfig(), False, False),
    "full_dense":           (full_feature_config(LOOKBACK, covariates=False), True, False),
    "full_dense_covariates":(full_feature_config(LOOKBACK, covariates=True), True, True),
    "dense_no_context":     (FeatureConfig(lag_samples=tuple(range(1, LOOKBACK + 1)),
                                           use_context=False), True, False),
}

df = load(OPERATOR)
profile = sampling_profile(df)

built = {}
for arm, (config, dense, covariates) in ARMS.items():
    assert_no_leakage(df, profile, config)      # gate: every arm, including the new block
    built[arm] = (build_features(df, profile, config), dense, covariates)

common = None
for (Xc, _), _, _ in built.values():
    common = Xc.index if common is None else common.intersection(Xc.index)
folds = rolling_origin(len(common), n_folds=N_FOLDS)
print(f"{OPERATOR.upper()}: {len(common)} rows common to all {len(ARMS)} arms, {N_FOLDS} folds")
for arm, ((Xc, _), _, _) in built.items():
    print(f"  {arm:24} {Xc.shape[1]:4} columns")


## What the channels actually are

Worth printing once, because the shape of the input is the whole point of this notebook.

In [ ]:

X_cov, y_cov = built["full_dense_covariates"][0]
spec = channel_window(X_cov, LOOKBACK)

print(f"{spec.n_channels} channels x {spec.lookback} lags, plus {len(spec.static)} static columns")
print()
print("channels:", ", ".join(spec.channels))
print()
print("static  :", ", ".join(spec.static))

window = spec.window(X_cov)
print(f"\nwindow tensor: {window.shape}   (rows, lookback, channels)")
print("channel 0 is demand, oldest observation first — position -1 is lag 1")


## Running the arms

Trees run in every arm. The univariate sequence models need a consecutive demand window,
so they sit out `full_sparse`. The covariate-aware variants (`*_cov`) only appear where
covariate channels exist. Roughly ten minutes on CPU.

In [ ]:

import time

def models_for(dense, covariates):
    models = list(default_point_models())
    if dense:
        models += sequence_models(lookback=LOOKBACK, epochs=EPOCHS)
    if covariates:
        models += covariate_sequence_models(lookback=LOOKBACK, epochs=EPOCHS)
    return models

per_fold_all, headline_predictions = [], None
for arm, ((Xc, yc), dense, covariates) in built.items():
    Xa, ya = Xc.loc[common], yc.loc[common]
    started = time.time()
    per_fold, predictions = run_backtest(
        Xa, ya, folds,
        models=models_for(dense, covariates),
        baselines=standard_baselines(ya.to_numpy(), profile.daily_period),
        season_lag=profile.daily_period,
    )
    per_fold["arm"] = arm
    per_fold_all.append(per_fold)
    if arm == "full_dense_covariates":
        headline_predictions = predictions
    print(f"{arm:24} {time.time() - started:5.0f}s")

per_fold = pd.concat(per_fold_all, ignore_index=True)
summary = (per_fold.groupby(["arm", "model"], as_index=False)["rmse"]
           .agg(rmse_mean="mean", rmse_std="std"))


## Arm against model

Read down a column to compare feature sets for one model; read across a row to compare
models on one feature set. `NaN` means the model could not read that arm's design matrix
at all, which is itself informative.

In [ ]:

table = summary.pivot(index="model", columns="arm", values="rmse_mean")
table = table[["lags_only", "full_sparse", "full_dense",
               "dense_no_context", "full_dense_covariates"]]
table.round(3).sort_values("full_dense_covariates")


### What to look for

Three comparisons carry the argument, and each is a subtraction between two columns:

- **`lags_only` → `full_dense`** — what the full feature set buys *once the window is
  deep*. Notebook 01's ablation found Fourier terms and context flags roughly neutral at
  four lags; this asks whether that survives at twenty-four.
- **`full_sparse` → `full_dense`** — lag depth at full features. Notebook 08 measured
  this at 18% on GP for a random forest.
- **`full_dense` → `full_dense_covariates`** — the covariate *history*, as opposed to a
  single contemporaneous value. This is the column that has never existed before, and
  the `*_cov` rows are the only models that can use it.

`dense_no_context` is the control that separates "more lags helped" from "context finally
reached a model that could use it".

## Does covariate history help, and is the difference real?

In [ ]:

pairs = []
for kind in ("cnn", "lstm", "gru", "rnn"):
    plain = table.loc[kind, "full_dense"]
    cov = table.loc[f"{kind}_cov", "full_dense_covariates"]
    pairs.append({"architecture": kind, "univariate_window": plain,
                  "with_covariates": cov, "change": (plain - cov) / plain})
pd.DataFrame(pairs).round(4)

In [ ]:

dm = dm_matrix(headline_predictions, horizon=1)
family = [m for m in table.index if m.endswith("_cov")] + \
         ["random_forest", "xgboost", "ridge", "persistence"]
shown = dm[dm["model_a"].isin(family) & dm["model_b"].isin(family)]
shown[["model_a", "model_b", "dm_stat", "p_value", "significant_fdr", "winner"]]


A `tie` here is the honest answer, not a missing result. Eight folds of roughly ninety
test points cannot resolve small differences, and this project reports that rather than
ranking on point estimates.

## Figure

In [ ]:

order = table["full_dense_covariates"].dropna().sort_values()
fig, ax = plt.subplots(figsize=(8.5, 4.6))
width = 0.38
positions = np.arange(len(order))
dense = table.loc[order.index, "full_dense"]
ax.barh(positions + width / 2, order.values, width,
        label="full features + covariate history", color="#2f6f9f")
ax.barh(positions - width / 2, dense.values, width,
        label="full features, demand window only", color="#9aa5b1")
ax.set_yticks(positions, order.index)
ax.axvline(table.loc["persistence", "full_dense_covariates"],
           color="#c0392b", ls="--", lw=1.3, label="persistence")
ax.invert_yaxis()
ax.set_xlabel("RMSE (mean over 8 folds)")
ax.set_title(f"{OPERATOR.upper()} — what the covariate channels are worth")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES / f"fig11_features_{OPERATOR}.png", dpi=200, bbox_inches="tight")


## Both operators at once

`experiments/run_full_features.py` runs everything above for GP and Robi and writes
`features_arm_matrix.csv`. If it has been run, the two are side by side here.

In [ ]:

path = RESULTS / "features_arm_matrix.csv"
if path.exists():
    both = pd.read_csv(path)
    display(both.pivot_table(index="model", columns=["operator", "arm"],
                             values="rmse_mean").round(3))
else:
    print("run experiments/run_full_features.py to fill this in")


## What to take away

1. **The bare window was a reproduction constraint, not a recommendation.** It existed so
   notebook 08 could isolate architecture from feature set, and it has now been lifted
   rather than inherited.
2. **Covariates are time-varying and enter as channels**, not as one flat vector. That is
   a modelling decision with a source (DeepAR; Temporal Fusion Transformer), not a
   convenience.
3. **Whether they earn their place is a measurement**, and the table above is it. A model
   that ignores its covariate channels fails a test gate, so a null result here means the
   covariates carry nothing on this trace — not that the wiring was wrong.

Notebook 10 asks the next question: given a deep window and the full feature set, does
the *architecture* matter?